# Scenarios

Demonstrate scenarios for satellite geolocation.

In [ ]:
from datetime import timedelta

from matplotlib import pyplot as plt
from skyfield.api import load
from skyfield.iokit import parse_tle_file

## Two Satellites

In [ ]:
ts = load.timescale()

with load.open("g16.tle", 'rb') as f:
    satellites = list(parse_tle_file(f, ts))

es_g16 = satellites[0]

with load.open("g19.tle", 'rb') as f:
    satellites = list(parse_tle_file(f, ts))

es_g19 = satellites[0]

es_g16, es_g19

In [ ]:
epoch_dt = es_g16.epoch.utc_datetime()

# Configuration for time increments
increment_minutes = 15
half_day = timedelta(hours=12)

# Generate list of datetimes from -0.5 days to +0.5 days from epoch_dt
start_dt = epoch_dt - half_day
end_dt = epoch_dt + half_day

dt_observables = []
observe_dt = start_dt
while observe_dt <= end_dt:
    dt_observables.append(observe_dt)
    observe_dt += timedelta(minutes=increment_minutes)

print(f"Generated {len(dt_observables)} datetime objects.")

Two antennas will be pointed to the respective satellites. We will again use a known emitters, the [Montana PBS](https://en.wikipedia.org/wiki/Montana_PBS), and the PBS Network Operations Center (NOC).

In [ ]:
from sat_geo_solver.scenarios import TwoSat

cos_ll = [+38.4, -104.82]
cos_ll_2 = [+38.4001, -104.8201]
pbs_noc_ll = [+38.8622183, -77.0504056, 63]
montana_pbs_ll = [+45.6655208, -111.0514232, 271]

two_sat = TwoSat(
    primary_satellite=es_g16,
    secondary_satellite=es_g19,
    at=dt_observables,
    primary_receiver=cos_ll,
    secondary_receiver=cos_ll_2,
)

### Differential Time Offset

Demonstrate differential time offset calculation between two satellites over time.

In [ ]:
primary_reference_signal_path = two_sat.primary.light_seconds(pbs_noc_ll) + two_sat.primary.light_seconds(cos_ll)
secondary_reference_signal_path = two_sat.secondary.light_seconds(pbs_noc_ll) + two_sat.secondary.light_seconds(cos_ll_2)

In [ ]:
plt.plot(primary_reference_signal_path, label="Primary")
plt.plot(secondary_reference_signal_path, label="Secondary")
plt.ylabel("Light Time (s)")
plt.title("Reference")
plt.legend()
plt.grid();

In [ ]:
dto_ref = two_sat.dto(from_pos=pbs_noc_ll)

In [ ]:
plt.plot(dto_ref)
plt.ylabel("Differential Time Offset (s)")
plt.title("Reference")
plt.grid();

In [ ]:
primary_target_signal_path = two_sat.primary.light_seconds(montana_pbs_ll) + two_sat.primary.light_seconds(cos_ll)
secondary_target_signal_path = two_sat.secondary.light_seconds(montana_pbs_ll) + two_sat.secondary.light_seconds(cos_ll_2)

In [ ]:
plt.plot(primary_target_signal_path, label="Primary")
plt.plot(secondary_target_signal_path, label="Secondary")
plt.ylabel("Light Time (s)")
plt.legend()
plt.grid();

In [ ]:
dto_tar = two_sat.dto(from_pos=montana_pbs_ll)

In [ ]:
plt.plot(dto_tar)
plt.ylabel("Differential Time Offset (s)")
plt.title("Target")
plt.grid();

## Different Frequency Offset

Demonstrate differential frequency offset calculation between two satellites over time.

In [ ]:
translation_frequency = 2.3e9
montana_pbs_freq = 11.806e9
montana_pbs_uplink = montana_pbs_freq + translation_frequency
pbs_downlink = 12.14e9
pbs_uplink = pbs_downlink + translation_frequency

In [ ]:
primary_reference_signal_doppler = two_sat.primary.downlink_received_frequency(from_pos=pbs_noc_ll, to_pos=cos_ll, uplink=pbs_uplink, translation_frequency=translation_frequency)
secondary_target_signal_doppler = two_sat.secondary.downlink_received_frequency(from_pos=pbs_noc_ll, to_pos=cos_ll_2, uplink=pbs_uplink, translation_frequency=translation_frequency)

In [ ]:
plt.plot(primary_reference_signal_doppler, label="Primary")
plt.plot(secondary_target_signal_doppler, label="Secondary")
plt.ylabel("Doppler Shift (Hz)")
plt.title("Reference")
plt.legend()
plt.grid();

In [ ]:
dfo_ref = two_sat.dfo(from_pos=pbs_noc_ll, uplink=pbs_uplink, translation_frequency=translation_frequency)

In [ ]:
plt.plot(dfo_ref)
plt.ylabel("Different Frequency Offset (Hz)")
plt.title("Reference")
plt.grid();

In [ ]:
primary_target_signal_doppler = two_sat.primary.downlink_received_frequency(from_pos=montana_pbs_ll, to_pos=cos_ll, uplink=montana_pbs_uplink, translation_frequency=translation_frequency)
secondary_target_signal_doppler = two_sat.secondary.downlink_received_frequency(from_pos=montana_pbs_ll, to_pos=cos_ll_2, uplink=montana_pbs_uplink, translation_frequency=translation_frequency)

In [ ]:
plt.plot(primary_target_signal_doppler, label="Primary")
plt.plot(secondary_target_signal_doppler, label="Secondary")
plt.ylabel("Doppler Shift (Hz)")
plt.title("Target")
plt.legend()
plt.grid();

In [ ]:
dfo_tar = two_sat.dfo(from_pos=montana_pbs_ll, uplink=pbs_uplink, translation_frequency=translation_frequency)

In [ ]:
plt.plot(dfo_tar)
plt.ylabel("Different Frequency Offset (Hz)")
plt.title("Target")
plt.grid();

### TDOA and FDOA

By taking the difference between target and reference, we can calculate the time difference of arrival (TDOA) and frequency difference of arrival (FDOA).

In [ ]:
tdoa = dto_tar - dto_ref
plt.plot(tdoa)
plt.title("TDOA")
plt.ylabel("Time Difference of Arrival (s)")
plt.grid();


In [ ]:
fdoa = dfo_tar - dfo_ref
plt.plot(fdoa)
plt.title("FDOA")
plt.ylabel("Frequency Difference of Arrival (Hz)")
plt.grid();